In [2]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Tạo thư mục ở project root thay vì trong notebooks/
os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

In [3]:
import sys
# Lùi lại 1 thư mục để truy cập vào src/models/
sys.path.append('../src/models')

from model_config import INPUT_SHAPE, NUM_CLASSES
from architectures import build_lstm_model, build_bilstm_model, build_cnn1d_model

print(f"Cấu hình từ T03: INPUT_SHAPE={INPUT_SHAPE}, NUM_CLASSES={NUM_CLASSES}")

# Lùi lại 1 thư mục để truy cập vào data/holistic/
data_dir = "../data/holistic/" 
X_train = np.load(f"{data_dir}X_train_aug.npy")

Cấu hình từ T03: INPUT_SHAPE=(30, 534), NUM_CLASSES=30


In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_and_print(model, model_name, X_test, y_test):
    # Lấy dự đoán từ mô hình
    y_pred_probs = model.predict(X_test, verbose=0)
    # Chuyển đổi xác suất thành nhãn (Label Encoding)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Tính toán các chỉ số đánh giá
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # In kết quả theo format định dạng sẵn
    print("=================================================================")
    print(f"           {model_name.upper()} EVALUATION RESULTS (TEST SET)")
    print("=================================================================")
    print(f"Accuracy:          {acc * 100:.2f}%")
    print(f"Macro Precision:   {precision:.4f}")
    print(f"Macro Recall:      {recall:.4f}")
    print(f"Macro F1-score:    {f1:.4f}")
    print("=================================================================")

In [5]:
# Đã thêm ../ để lùi ra thư mục gốc
data_dir = "../data/holistic/" 

X_train = np.load(f"{data_dir}X_train_aug.npy")
y_train = np.load(f"{data_dir}y_train_aug.npy")
X_val = np.load(f"{data_dir}X_val.npy")
y_val = np.load(f"{data_dir}y_val.npy")
X_test = np.load(f"{data_dir}X_test.npy")
y_test = np.load(f"{data_dir}y_test.npy")

print("\n--- KIỂM TRA SHAPE DỮ LIỆU ---")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

# Đảm bảo nhãn đang ở dạng mảng 1 chiều (Label Encoding)
assert len(y_train.shape) == 1, "LỖI: Nhãn không phải Label Encoding 1 chiều!"
print("\n-> [OK] Dữ liệu đã sẵn sàng để huấn luyện đa mô hình!")


--- KIỂM TRA SHAPE DỮ LIỆU ---
X_train shape: (3000, 30, 534)
y_train shape: (3000,)
X_val shape: (210, 30, 534)
y_val shape: (210,)

-> [OK] Dữ liệu đã sẵn sàng để huấn luyện đa mô hình!


In [7]:
# CẤU HÌNH HUẤN LUYỆN CHUNG
EPOCHS = 50
BATCH_SIZE = 32

In [8]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN LSTM ---")
lstm_model = build_lstm_model()

lstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Thêm ../ để lưu đúng vào thư mục gốc
    ModelCheckpoint('../models/lstm_model.h5', monitor='val_loss', save_best_only=True)
]

lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=lstm_callbacks,
    verbose=1
)

# ĐÃ SỬA: Thêm ../ để lưu đúng vào thư mục gốc
with open('../results/lstm_history.json', 'w') as f:
    json.dump(lstm_history.history, f)

print(f"\n[*] LSTM training completed.")
print(f"[*] Final validation accuracy: {lstm_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/lstm_model.h5")


--- XÂY DỰNG & HUẤN LUYỆN LSTM ---
Epoch 1/50


94/94 [==============================] - 10s 61ms/step - loss: 2.6954 - accuracy: 0.1990 - val_loss: 2.0603 - val_accuracy: 0.3333
Epoch 2/50
 1/94 [..............................] - ETA: 6s - loss: 2.2945 - accuracy: 0.2812

d:\anaconda3\envs\sign_lang_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - 5s 54ms/step - loss: 1.7707 - accuracy: 0.4323 - val_loss: 1.5431 - val_accuracy: 0.5286
Epoch 3/50
94/94 [==============================] - 5s 55ms/step - loss: 1.4068 - accuracy: 0.5537 - val_loss: 1.2136 - val_accuracy: 0.6000
Epoch 4/50
94/94 [==============================] - 5s 57ms/step - loss: 0.9874 - accuracy: 0.6773 - val_loss: 1.0714 - val_accuracy: 0.6429
Epoch 5/50
94/94 [==============================] - 5s 51ms/step - loss: 0.8052 - accuracy: 0.7307 - val_loss: 1.1315 - val_accuracy: 0.6714
Epoch 6/50
94/94 [==============================] - 5s 54ms/step - loss: 0.5767 - accuracy: 0.8183 - val_loss: 0.9876 - val_accuracy: 0.7571
Epoch 7/50
94/94 [==============================] - 6s 62ms/step - loss: 0.4464 - accuracy: 0.8560 - val_loss: 0.9051 - val_accuracy: 0.7619
Epoch 8/50
94/94 [==============================] - 5s 57ms/step - loss: 0.4434 - accuracy: 0.8557 - val_loss: 0.7348 - val_accuracy: 0.8048
Epoch 9/50
94/94 [======

In [11]:
# Gọi hàm xuất kết quả cho LSTM
evaluate_and_print(lstm_model, "LSTM", X_test, y_test)

           LSTM EVALUATION RESULTS (TEST SET)
Accuracy:          85.31%
Macro Precision:   0.8763
Macro Recall:      0.8542
Macro F1-score:    0.8494


In [12]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN BiLSTM ---")
bilstm_model = build_bilstm_model()

# ĐÃ XÓA: dòng bilstm_model.compile(...) vì đã được gọi bên trong hàm build_bilstm_model()

bilstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Thêm ../ vào đường dẫn
    ModelCheckpoint('../models/bilstm_model.h5', monitor='val_loss', save_best_only=True)
]

bilstm_history = bilstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=bilstm_callbacks,
    verbose=1
)

# ĐÃ SỬA: Thêm ../ vào đường dẫn
with open('../results/bilstm_history.json', 'w') as f:
    json.dump(bilstm_history.history, f)

print(f"\n[*] BiLSTM training completed.")
print(f"[*] Final validation accuracy: {bilstm_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/bilstm_model.h5")


--- XÂY DỰNG & HUẤN LUYỆN BiLSTM ---
Epoch 1/50
94/94 [==============================] - 14s 99ms/step - loss: 2.8331 - accuracy: 0.2280 - val_loss: 2.2371 - val_accuracy: 0.3905
Epoch 2/50
 1/94 [..............................] - ETA: 9s - loss: 1.7496 - accuracy: 0.4062

d:\anaconda3\envs\sign_lang_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - 10s 101ms/step - loss: 1.7510 - accuracy: 0.4563 - val_loss: 1.4033 - val_accuracy: 0.5571
Epoch 3/50
94/94 [==============================] - 10s 108ms/step - loss: 1.1672 - accuracy: 0.6333 - val_loss: 1.1674 - val_accuracy: 0.6238
Epoch 4/50
94/94 [==============================] - 11s 116ms/step - loss: 0.8055 - accuracy: 0.7420 - val_loss: 1.0372 - val_accuracy: 0.6952
Epoch 5/50
94/94 [==============================] - 10s 104ms/step - loss: 0.6963 - accuracy: 0.7857 - val_loss: 1.0653 - val_accuracy: 0.6952
Epoch 6/50
94/94 [==============================] - 10s 103ms/step - loss: 0.6166 - accuracy: 0.8113 - val_loss: 0.9471 - val_accuracy: 0.7476
Epoch 7/50
94/94 [==============================] - 11s 114ms/step - loss: 0.4345 - accuracy: 0.8573 - val_loss: 0.8957 - val_accuracy: 0.7571
Epoch 8/50
94/94 [==============================] - 10s 106ms/step - loss: 0.3157 - accuracy: 0.9043 - val_loss: 0.8298 - val_accuracy: 0.7952
Epoch 9/50

In [13]:
evaluate_and_print(bilstm_model, "BiLSTM", X_test, y_test)

           BILSTM EVALUATION RESULTS (TEST SET)
Accuracy:          84.83%
Macro Precision:   0.8562
Macro Recall:      0.8427
Macro F1-score:    0.8394


In [14]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN 1D-CNN ---")
cnn1d_model = build_cnn1d_model()

# ĐÃ XÓA: dòng cnn1d_model.compile(...) vì đã được gọi bên trong hàm build_cnn1d_model()

cnn1d_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Thêm ../ vào đường dẫn để lưu đúng thư mục gốc
    ModelCheckpoint('../models/cnn1d_model.h5', monitor='val_loss', save_best_only=True)
]

cnn1d_history = cnn1d_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cnn1d_callbacks,
    verbose=1
)

# ĐÃ SỬA: Thêm ../ vào đường dẫn
with open('../results/cnn1d_history.json', 'w') as f:
    json.dump(cnn1d_history.history, f)

print(f"\n[*] 1D-CNN training completed.")
print(f"[*] Final validation accuracy: {cnn1d_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/cnn1d_model.h5")


--- XÂY DỰNG & HUẤN LUYỆN 1D-CNN ---
Epoch 1/50
94/94 [==============================] - 3s 14ms/step - loss: 2.4538 - accuracy: 0.2880 - val_loss: 1.6782 - val_accuracy: 0.4952
Epoch 2/50
10/94 [==>...........................] - ETA: 1s - loss: 1.5193 - accuracy: 0.5437

d:\anaconda3\envs\sign_lang_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - 2s 17ms/step - loss: 1.1708 - accuracy: 0.6513 - val_loss: 1.2196 - val_accuracy: 0.6143
Epoch 3/50
94/94 [==============================] - 1s 14ms/step - loss: 0.7414 - accuracy: 0.7900 - val_loss: 0.9072 - val_accuracy: 0.7381
Epoch 4/50
94/94 [==============================] - 1s 13ms/step - loss: 0.5189 - accuracy: 0.8427 - val_loss: 0.7385 - val_accuracy: 0.7762
Epoch 5/50
94/94 [==============================] - 1s 13ms/step - loss: 0.3682 - accuracy: 0.9013 - val_loss: 0.5859 - val_accuracy: 0.8333
Epoch 6/50
94/94 [==============================] - 1s 13ms/step - loss: 0.3001 - accuracy: 0.9137 - val_loss: 0.5787 - val_accuracy: 0.8524
Epoch 7/50
94/94 [==============================] - 2s 17ms/step - loss: 0.2470 - accuracy: 0.9353 - val_loss: 0.5756 - val_accuracy: 0.8429
Epoch 8/50
94/94 [==============================] - 1s 12ms/step - loss: 0.2030 - accuracy: 0.9487 - val_loss: 0.5609 - val_accuracy: 0.8429
Epoch 9/50
94/94 [======

In [15]:
evaluate_and_print(cnn1d_model, "1D-CNN", X_test, y_test)

           1D-CNN EVALUATION RESULTS (TEST SET)
Accuracy:          86.73%
Macro Precision:   0.8769
Macro Recall:      0.8581
Macro F1-score:    0.8598


In [16]:
from tensorflow.keras.models import load_model

print("\n--- KIỂM TRA MODEL ĐÃ LƯU ---")
saved_models = {
    # ĐÃ SỬA: Thêm ../ vào trước tất cả các đường dẫn
    "LSTM": "../models/lstm_model.h5",
    "BiLSTM": "../models/bilstm_model.h5",
    "1D-CNN": "../models/cnn1d_model.h5"
}

# Lấy 1 sample từ tập X_test để chạy thử
sample = X_test[:1]

for name, path in saved_models.items():
    if os.path.exists(path):
        # Tải mô hình lên
        test_model = load_model(path)
        
        # Cho mô hình dự đoán thử 1 mẫu
        pred = test_model.predict(sample, verbose=0)
        
        # Kiểm tra xem đầu ra có đúng là 30 class không
        assert pred.shape == (1, NUM_CLASSES), f"Lỗi shape ở {name}"
        
        print(f"-> [OK] {name} tải thành công. Output Shape: {pred.shape}")
    else:
        print(f"-> [LỖI] Không tìm thấy file {path}")


--- KIỂM TRA MODEL ĐÃ LƯU ---
-> [OK] LSTM tải thành công. Output Shape: (1, 30)
-> [OK] BiLSTM tải thành công. Output Shape: (1, 30)
-> [OK] 1D-CNN tải thành công. Output Shape: (1, 30)
